<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.4-vit-and-clip/notebooks/exercises-4.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.4 — Vision Transformers (ViT) & CLIP
Netsetos GenAI Course v2.0 | Module 4

ViT architecture, CLIP contrastive learning, zero-shot classification, image search, SD integration, India applications


In [ ]:
# Setup
# pip install torch transformers timm faiss-cpu pillow -q
import torch
print(f'PyTorch: {torch.__version__}')


## Ex 1: ViT Patch Visualization


In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Create or load a 224x224 image
img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)

# Split into 16x16 patches
patch_size = 16
patches = []
for i in range(0, 224, patch_size):
    for j in range(0, 224, patch_size):
        patches.append(img[i:i+patch_size, j:j+patch_size])

print(f'Image: {img.shape}')
print(f'Patches: {len(patches)} patches of {patches[0].shape}')
print(f'Grid: {224//patch_size}×{224//patch_size} = {len(patches)}')
print(f'Each patch flattened: {patch_size*patch_size*3} = 768 dims')


## Ex 2: CLIP Zero-Shot Classification


In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch

model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

# Create test image (or load your own)
image = Image.new('RGB', (224, 224), (255, 100, 0))  # Orange

categories = ['fire', 'ocean', 'forest', 'sunset', 'snow']
inputs = processor(text=categories, images=image, return_tensors='pt', padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits_per_image
    probs = logits.softmax(dim=-1)[0]

for cat, prob in sorted(zip(categories, probs), key=lambda x: -x[1]):
    print(f'  {cat:15s}: {prob:.3f} {"✅" if prob > 0.3 else ""}')
print('\nZero-shot: no training needed. Just describe categories in text.')


## Ex 3: Contrastive Loss Implementation


In [ ]:
import torch
import torch.nn.functional as F

def clip_loss(image_embs, text_embs, temperature=0.07):
    # L2 normalize
    img = F.normalize(image_embs, dim=-1)
    txt = F.normalize(text_embs, dim=-1)
    # N×N similarity matrix
    logits = (img @ txt.T) / temperature
    # Labels: diagonal = correct pairs
    labels = torch.arange(len(logits))
    # Symmetric loss
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2

# Test with batch of 8
img_emb = torch.randn(8, 512)
txt_emb = torch.randn(8, 512)
loss = clip_loss(img_emb, txt_emb)
print(f'Contrastive loss: {loss:.4f}')
print(f'Similarity matrix shape: {(img_emb @ txt_emb.T).shape}')
print('Diagonal = correct pairs. Off-diagonal = negatives.')


## Ex 4: CLIP Limitation Tests


In [ ]:
# Test CLIP's known failure modes
from transformers import CLIPModel, CLIPProcessor
import torch

model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

# Test: spatial relationships
texts = ['a cat sitting on a dog', 'a dog sitting on a cat']
image = Image.new('RGB', (224, 224), (128, 128, 128))

inputs = processor(text=texts, images=image, return_tensors='pt', padding=True)
with torch.no_grad():
    outputs = model(**inputs)
    sims = outputs.logits_per_image[0]

print('Spatial relationship test:')
for t, s in zip(texts, sims):
    print(f'  "{t}": {s:.3f}')
print(f'Difference: {abs(sims[0]-sims[1]):.3f} (should be large, but is small!)')
print('\nCLIP treats these as bag-of-words → similar scores')


## Ex 5: Image Search with FAISS


In [ ]:
import numpy as np
# import faiss  # pip install faiss-cpu

# Simulate CLIP embeddings for 100 images
np.random.seed(42)
num_images = 100
dim = 512  # ViT-B/32

# Mock embeddings (in production: clip_model.get_image_features())
image_embeddings = np.random.randn(num_images, dim).astype(np.float32)
# L2 normalize (CRITICAL!)
image_embeddings /= np.linalg.norm(image_embeddings, axis=1, keepdims=True)

# Build FAISS index
# index = faiss.IndexFlatIP(dim)  # Inner product = cosine for normalized
# index.add(image_embeddings)

# Search with text query
query = np.random.randn(1, dim).astype(np.float32)
query /= np.linalg.norm(query)
# distances, indices = index.search(query, k=5)

print(f'Indexed {num_images} images ({dim}-dim embeddings)')
print(f'Query shape: {query.shape}')
print('FAISS search: <1ms for 100K images, ~10ms for 1M')
print('\nVector DB options:')
for db, use in [('FAISS','Prototyping (free)'),('Qdrant','Production <5M'),('Milvus','Billion-scale'),('Weaviate','E-commerce hybrid')]:
    print(f'  {db:12s}: {use}')


## Ex 6: timm Pre-trained ViTs


In [ ]:
import timm

# List available ViT models
vit_models = timm.list_models('vit_*', pretrained=True)
print(f'Available ViT models: {len(vit_models)}')
for m in vit_models[:5]: print(f'  {m}')

# Create model for classification
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=10)
print(f'\nViT-B/16 for 10 classes: {sum(p.numel() for p in model.parameters()):,} params')

# Feature extraction (no classification head)
model_feat = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
print(f'Feature extractor output: {model_feat(torch.randn(1, 3, 224, 224)).shape}')


## Ex 7: CLIP Score for SD Evaluation


In [ ]:
# from torchmetrics.multimodal.clip_score import CLIPScore
# metric = CLIPScore(model_name_or_path='openai/clip-vit-large-patch14')
# score = metric(image_tensor, 'a beautiful sunset over mountains')
# print(f'CLIP Score: {score:.1f}')  # Typical SD output: ~35

print('CLIP Score = max(100 × cos(E_image, E_text), 0)')
print('Typical SD 1.5 output: ~35')
print('Higher = better text-image alignment')
print('\nCaveats:')
print('  - Biased toward CLIP\'s training distribution')
print('  - Can\'t capture fine-grained quality')
print('  - Different phrasings → different scores')
print('  - Use alongside FID + Aesthetic Score')


## Ex 8: India Cost Calculator


In [ ]:
providers = {
    'AWS (Singapore)': {'gpu': 'L4', 'cost_hr': 250, 'emb_per_sec': 800},
    'E2E Networks': {'gpu': 'L4', 'cost_hr': 60, 'emb_per_sec': 800},
    'Cyfuture AI': {'gpu': 'L40S', 'cost_hr': 61, 'emb_per_sec': 1200},
}

for vol in [100_000, 1_000_000, 10_000_000]:
    print(f'\n{vol:,} CLIP inferences:')
    for name, p in providers.items():
        hours = vol / p['emb_per_sec'] / 3600
        cost = hours * p['cost_hr']
        print(f'  {name:20s}: {hours:.1f} hrs × ₹{p["cost_hr"]}/hr = ₹{cost:,.0f} (₹{cost/vol*1e6:.0f}/million)')

print('\nKey: Indian providers 40-70% cheaper than hyperscalers')
print('Fine-tune CLIP on Indian product data for 50%+ improvement')


---
## Recap
ViT = images as token sequences (patches → Transformer). CLIP = contrastive learning aligns images and text in shared embedding space. SigLIP = better CLIP (sigmoid loss, no global normalization). DINOv3 = best self-supervised features.

CLIP powers SD (text encoder + safety checker + CLIP Score + IP-Adapter). Five failure modes: counting, spatial, compositionality, typographic attacks, bias. Production: CLIP + FAISS/vector DB for search. India: Qure.ai (medical), SatSure (agriculture), Flipkart (e-commerce), ₹6-15/million inferences.
